# Benchmark Analysis Report

使用 data_loader 加载数据，分析成功/失败案例

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from core.data_loader import (
    load_results, load_problems, load_ratings, merge_all,
    get_failed_cases, get_low_score_cases, summarize_by_recipe
)

# 配置 run_dir（运行时替换）
RUN_DIR = Path("outputs/run_YYYYMMDD_HHMMSS")  # 替换为实际路径

In [ ]:
results = load_results(RUN_DIR)
problems = load_problems(RUN_DIR)
ratings = load_ratings(RUN_DIR)
df = merge_all(RUN_DIR)

print(f"Results: {len(results)} rows")
print(f"Problems: {len(problems)} rows")
print(f"Ratings: {len(ratings)} rows")
print(f"Merged: {len(df)} rows")

## 失败案例分析

In [ ]:
failed = get_failed_cases(df)
print(f"Failed cases: {len(failed)} / {len(df)}")

if len(failed) > 0:
    # 按 fail_type 统计
    fail_by_type = failed.groupby('fail_type').size().reset_index(name='count')
    fig = px.bar(fail_by_type, x='fail_type', y='count', title='Failure Types Distribution')
    fig.show()
    
    # 展示典型失败案例
    display(failed[['problem_id', 'recipe_name', 'seed', 'fail_type', 'message']].head(10))

## 低分案例分析

In [ ]:
low_score = get_low_score_cases(df, threshold=2.0)
high_score = df[df['rating'] >= 3.0]

print(f"Low score cases (< 2.0): {len(low_score)}")
print(f"High score cases (>= 3.0): {len(high_score)}")

if len(low_score) > 0 and len(high_score) > 0:
    # 对比展示
    comparison = pd.DataFrame({
        'group': ['Low Score', 'High Score'],
        'mean_solve_time': [low_score['solve_time_s'].mean(), high_score['solve_time_s'].mean()],
        'count': [len(low_score), len(high_score)]
    })
    display(comparison)

## Recipe 对比分析

In [ ]:
recipe_summary = summarize_by_recipe(df)
display(recipe_summary)

# 成功率 vs 平均评分散点图
if 'mean_rating' in recipe_summary.columns:
    fig = px.scatter(
        recipe_summary, 
        x='success_rate', 
        y='mean_rating', 
        size='count',
        hover_name='recipe_name',
        title='Success Rate vs Mean Rating by Recipe'
    )
    fig.show()

## 聚类分析框架（TODO）

In [ ]:
# TODO: 添加聚类分析
# 可以使用 sklearn.cluster 对 solve_time, success_rate, rating 等进行聚类
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans